# Streaming Signal Decomposition — Complete Feature Demo

**DACS Bachelor Thesis, Maastricht University, Spring 2026**

This notebook is a self-contained, end-to-end demonstration of every implemented feature
of the Streaming SSD framework. Execute top-to-bottom with *Kernel → Restart & Run All*.

| Section | Topic |
|---------|-------|
| 0 | Imports & Setup |
| 1 | Signal Generation & Visualisation |
| 2 | Offline SSD Baseline |
| 3 | Randomised SSD (rSSD) |
| 4 | Sliding-Window Decomposition — Naive |
| 5 | Component Matching & Temporal Consistency |
| 6 | Incremental SVD Update |
| 7 | Full Streaming Pipeline — End-to-End |
| 8 | Quantitative Evaluation |
| 9 | Real-World Application Demo |
| 10 | Summary & Reproducibility |

## Section 0 — Imports & Setup

All imports are declared here. Shared constants (window length, stride, SNR levels, etc.)
are defined in one cell so every downstream section reads the same values. A global
wall-clock timer is started for the Section 10 execution-time report.

In [67]:
import sys, os, json, time, warnings, tracemalloc, importlib
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

# ---- signal generators ---------------------------------------------------
from experiments.synthetic.generators import (
    two_sinusoids, chirp_plus_sinusoid, rossler, component_onset, n_sinusoids,
)

# ---- decomposition engines -----------------------------------------------
import src.engines as _eng_pkg
importlib.reload(_eng_pkg)
from src.engines import (
    DecompositionEngine, get_engine,
    SSD, SSA, IncrementalSSD, RankOneIncrementalSSD,
    RankOneUpdater,
    build_trajectory_matrix, svd_decompose, diagonal_averaging, auto_ssa,
    rsvd,
)
from src.engines.ssd_optimized import OptimizedSSD
from src.engines import _REGISTRY
from src.engines.svd_update import _build_hankel

# ---- metrics -------------------------------------------------------------
from src.metrics.similarity import d_corr, d_freq, w_correlation, subspace_angle
from src.metrics.stability import (
    qrf, nmse, energy_continuity, singular_value_drift,
    dominant_frequency, freq_drift_aggregate, matching_confidence,
)

# ---- streaming pipeline --------------------------------------------------
from src.streaming.window_manager import WindowManager
from src.streaming.component_matcher import ComponentMatcher
from src.streaming.trajectory_store import TrajectoryStore

# ---- visualization -------------------------------------------------------
from src.visualization import (
    plot_decomposition, plot_trajectory_overlay, plot_component_spectra,
    plot_matching_graph, plot_metrics_over_windows, plot_pipeline_dashboard,
    plot_window_reconstruction, plot_window_grid, plot_nmse_over_time,
)
from src.visualization.plot_metrics import plot_metrics

# ---- experiment runner ---------------------------------------------------
from experiments.run_experiment import build_pipeline, run as run_experiment

# ==========================================================================
# SHARED CONSTANTS  (change here — affects all sections)
# ==========================================================================
FS         = 1000.0   # sampling frequency [Hz]
SEED       = 42       # global RNG seed — set once, never changed
N_SIGNAL   = 3000     # default signal length [samples]  (~3 s at 1 kHz)
WINDOW_LEN = 300      # streaming window length [samples] — ~10 cycles of f=10 Hz
STRIDE     = 150      # stride / hop [samples]  → 50 % overlap
NMSE_THR   = 0.01     # SSD NMSE stopping threshold (quality/speed trade-off)
MAX_ITER   = 20       # max SSD extraction iterations per window
MAX_COMP   = 30       # max simultaneous trajectory slots in TrajectoryStore
PALETTE    = plt.rcParams["axes.prop_cycle"].by_key()["color"]  # consistent colours

np.random.seed(SEED)
RESULTS_DIR = Path("../results/demo_run")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Notebook-level wall-clock start  (used in Section 10)
_T0_NOTEBOOK = time.perf_counter()

print("Setup complete.")
print(f"Results dir  : {RESULTS_DIR.resolve()}")
print(f"Engines      : {sorted(_REGISTRY.keys())}")

Setup complete.
Results dir  : /Users/sachaloeb/Documents/StreamingSignalDecompositionRTDA/results/demo_run
Engines      : ['ssa', 'ssd', 'ssd_incremental', 'ssd_optimized', 'ssd_rank1']


## Section 1 — Signal Generation & Visualisation

We generate **four** synthetic test signals covering distinct difficulty levels:

| ID | Generator | Character |
|----|-----------|----------|
| A  | `two_sinusoids` | Stationary, two well-separated spectral lines |
| B  | `chirp_plus_sinusoid` | Non-stationary chirp + stationary tone — primary signal |
| C  | hand-crafted | Trend + oscillation + AWGN (SNR 15 dB) |
| D  | `n_sinusoids` | Five-component mixture at SNR 20 dB |

Signal B is used as the **primary signal** in Sections 2–7. Signals A, C, D appear
in the benchmark (Section 8).

> **No real-world datasets are shipped with the repository.** Section 9 therefore applies
> the full pipeline to the Rössler attractor — a chaotic dynamical system whose x-component
> exhibits broad-band, non-stationary spectral structure that closely mimics IoT sensor data.

In [68]:
t = np.arange(N_SIGNAL) / FS   # shared time axis [s]

# A — stationary multi-component
sig_A = two_sinusoids(
    N=N_SIGNAL, f1=20.0, f2=80.0, A1=1.0, A2=0.7,
    fs=FS, snr_db=None, seed=SEED,
)

# B — non-stationary (primary signal used downstream)
sig_B = chirp_plus_sinusoid(
    N=N_SIGNAL, f_sin=50.0, f_start=10.0, f_end=150.0,
    fs=FS, snr_db=20.0, seed=SEED,
)

# C — trend + oscillation + AWGN at 15 dB
_rng_c = np.random.default_rng(SEED + 1)
_trend  = 0.5 * np.linspace(-1.0, 1.0, N_SIGNAL)
_osc    = np.sin(2 * np.pi * 5.0 * t)
_clean_c = _trend + _osc
_noise_power_c = np.var(_clean_c) / 10 ** (15.0 / 10)
sig_C = _clean_c + _rng_c.normal(0, np.sqrt(_noise_power_c), N_SIGNAL)

# D — n_sinusoids: five spectral components
sig_D = n_sinusoids(
    N=N_SIGNAL,
    frequencies=[10.0, 30.0, 60.0, 110.0, 180.0],
    amplitudes=[1.0, 0.8, 0.6, 0.4, 0.3],
    fs=FS, snr_db=20.0, seed=SEED,
)

signal = sig_B   # <-- primary signal for all downstream sections

for label, sig in [("A", sig_A), ("B", sig_B), ("C", sig_C), ("D", sig_D)]:
    print(f"  {label}: shape={sig.shape}  "
          f"mean={sig.mean():.3f}  std={sig.std():.3f}  "
          f"energy={np.dot(sig, sig):.1f}")

  A: shape=(3000,)  mean=0.000  std=0.863  energy=2235.0
  B: shape=(3000,)  mean=-0.002  std=1.009  energy=3052.6
  C: shape=(3000,)  mean=-0.001  std=0.763  energy=1748.4
  D: shape=(3000,)  mean=-0.003  std=1.067  energy=3416.5


In [69]:
descriptions = [
    "A — two_sinusoids  (f₁=20 Hz, f₂=80 Hz, stationary)",
    "B — chirp_plus_sinusoid  (10→150 Hz chirp + 50 Hz tone, SNR=20 dB)  ← primary",
    "C — trend + oscillation + AWGN  (5 Hz osc, SNR=15 dB)",
    "D — n_sinusoids  (5 components, SNR=20 dB)",
]
signals = [sig_A, sig_B, sig_C, sig_D]

fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=False)
for ax, sig, desc, color in zip(axes, signals, descriptions, PALETTE):
    ax.plot(t, sig, linewidth=0.7, color=color)
    ax.set_title(desc, fontsize=9)
    ax.set_ylabel("amplitude")

axes[-1].set_xlabel("time (s)")
fig.suptitle("Section 1 — Synthetic Test Signals", fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "01_signals.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: 01_signals.png")

Saved: 01_signals.png


## Section 2 — Offline SSD Baseline (Full-signal, Standard SVD)

**Algorithm:** Singular Spectrum Decomposition — Bonizzi, Karel, Meste & Peeters (2014).
*Advances in Adaptive Data Analysis.*

SSD iteratively extracts dominant spectral components from a signal by:
1. Building a **wrapped trajectory matrix** (each row is a circular shift of the signal)
2. **SVD decomposition** to expose the spectral structure
3. **Automatic eigentriple selection** using a Gaussian spectral model (FWHM criterion)
4. **Diagonal averaging** (modified for the wrapped formulation) to reconstruct the component
5. Subtracting the component and repeating until the residual NMSE falls below threshold

This section provides the **reference baseline** against which all streaming variants are compared.

In [70]:
# ── 2.1  Trajectory matrix construction ────────────────────────────────────

ssd_ref = SSD(fs=FS, nmse_threshold=NMSE_THR, max_iter=MAX_ITER)

# Automatic window-length selection: M ≈ 1.2 × fs / f_max  (clipped to [2, N//3])
M_auto = ssd_ref._choose_window_length(signal)
print(f"Automatic embedding dimension M = {M_auto}  (rule: 1.2 × fs / f_max)")

# Wrapped trajectory matrix  →  shape (M, N)
X_wrapped = SSD._build_trajectory_matrix(signal, M_auto)
print(f"Wrapped trajectory matrix: {X_wrapped.shape}  (M rows × N columns)")
assert X_wrapped.shape == (M_auto, N_SIGNAL)

# Standard Hankel matrix for side-by-side comparison
L_ssa = min(150, N_SIGNAL // 3)
X_hankel = build_trajectory_matrix(signal, L=L_ssa)
print(f"Standard Hankel matrix   : {X_hankel.shape}  (L rows × K columns)")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(X_wrapped[:, :250], aspect="auto", cmap="RdBu_r",
               vmin=-2, vmax=2, interpolation="nearest")
axes[0].set_title(f"Wrapped trajectory matrix  (Bonizzi 2014)\nM={M_auto}, first 250 cols")
axes[0].set_xlabel("column (time lag j)"); axes[0].set_ylabel("row (embedding lag i)")

axes[1].imshow(X_hankel[:50, :250], aspect="auto", cmap="RdBu_r",
               vmin=-2, vmax=2, interpolation="nearest")
axes[1].set_title(f"Standard Hankel matrix  (SSA formulation)\nL={L_ssa}, first 50 rows, 250 cols")
axes[1].set_xlabel("column"); axes[1].set_ylabel("row")

fig.suptitle("Section 2 — Trajectory Matrix Embedding", fontweight="bold")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "02_traj_matrix.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: 02_traj_matrix.png")

Automatic embedding dimension M = 23  (rule: 1.2 × fs / f_max)
Wrapped trajectory matrix: (23, 3000)  (M rows × N columns)
Standard Hankel matrix   : (150, 2851)  (L rows × K columns)
Saved: 02_traj_matrix.png


In [71]:
# ── 2.2  SVD step: singular spectrum ───────────────────────────────────────

U_ref, S_ref, Vt_ref = np.linalg.svd(X_wrapped, full_matrices=False)
print(f"SVD of trajectory matrix: U{U_ref.shape}, S{S_ref.shape}, Vt{Vt_ref.shape}")
print(f"Top-10 singular values: {np.round(S_ref[:10], 2)}")

# Cumulative energy explained
cumvar = np.cumsum(S_ref**2) / np.sum(S_ref**2) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

n_show = min(30, len(S_ref))
axes[0].bar(np.arange(1, n_show + 1), S_ref[:n_show], color="steelblue")
axes[0].set_yscale("log")
axes[0].set_title(f"Singular spectrum (top-{n_show} values, log scale)")
axes[0].set_xlabel("singular value index")
axes[0].set_ylabel("singular value (log)")

axes[1].plot(np.arange(1, len(cumvar[:50]) + 1), cumvar[:50], "o-",
             markersize=4, color="darkorange")
axes[1].axhline(95, linestyle="--", color="red", linewidth=1, label="95 % threshold")
axes[1].set_title("Cumulative energy explained (%)")
axes[1].set_xlabel("number of singular values retained")
axes[1].set_ylabel("% variance explained")
axes[1].legend()

fig.suptitle("Section 2 — SVD of Wrapped Trajectory Matrix", fontweight="bold")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "02_svd_spectrum.png", dpi=120, bbox_inches="tight")
plt.show()

SVD of trajectory matrix: U(23, 23), S(23,), Vt(23, 3000)
Top-10 singular values: [156.96 144.14  73.58  73.52  70.69  63.16  53.17  37.79  16.81   7.91]


In [72]:
# ── 2.3  Full SSD on all three signals — reconstruction quality table ───────

s2_results = {}
for name, sig in [("A-stationary", sig_A),
                   ("B-nonstat",   sig_B),
                   ("C-noisy",     sig_C)]:
    eng = SSD(fs=FS, nmse_threshold=NMSE_THR, max_iter=MAX_ITER)
    t0 = time.perf_counter()
    comps = eng.fit(sig)
    elapsed_ms = 1000 * (time.perf_counter() - t0)
    comps_only, residual = comps[:-1], comps[-1]
    recon = np.sum(comps_only, axis=0)
    # identity check: sum(components) + residual == signal
    assert np.allclose(recon + residual, sig, atol=1e-10), "Reconstruction identity violated"
    n_val = nmse(sig - recon, sig)
    q_val = qrf(sig, recon)
    s2_results[name] = dict(comps=comps_only, residual=residual,
                            recon=recon, nmse=n_val, qrf=q_val,
                            n_comps=len(comps_only), time_ms=elapsed_ms)
    print(f"  {name:15s}: {len(comps_only):2d} comps  "
          f"NMSE={n_val:.5f}  QRF={q_val:.2f} dB  t={elapsed_ms:.0f} ms")

# Primary offline result (signal B)
ssd_comps   = s2_results["B-nonstat"]["comps"]
ssd_residual = s2_results["B-nonstat"]["residual"]
ssd_recon    = s2_results["B-nonstat"]["recon"]

  A-stationary   :  2 comps  NMSE=0.00000  QRF=79.92 dB  t=62 ms
  B-nonstat      :  7 comps  NMSE=0.00838  QRF=20.77 dB  t=154 ms
  C-noisy        :  3 comps  NMSE=0.00958  QRF=20.19 dB  t=165 ms


In [73]:
# ── 2.4  Visualise SSD decomposition (signal B) ─────────────────────────────

# (a) Full decomposition stacked plot
plot_decomposition(
    signal, ssd_comps, residual=ssd_residual, fs=FS,
    title="Section 2 — Offline SSD Decomposition (Signal B: chirp+sinusoid)",
    save_path=str(RESULTS_DIR / "02_ssd_decomp.png"),
)

# (b) Original vs reconstruction
fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
axes[0].plot(t, signal, linewidth=0.7, color="steelblue", label="original")
axes[0].plot(t, ssd_recon, linewidth=0.8, color="darkorange",
             alpha=0.85, label="SSD reconstruction")
axes[0].set_ylabel("amplitude")
axes[0].set_title(f"Original vs SSD Reconstruction  |  "
                  f"NMSE={s2_results['B-nonstat']['nmse']:.5f}  "
                  f"QRF={s2_results['B-nonstat']['qrf']:.2f} dB")
axes[0].legend(fontsize=9)

axes[1].plot(t, signal - ssd_recon, linewidth=0.7, color="crimson", label="residual")
axes[1].set_ylabel("amplitude")
axes[1].set_xlabel("time (s)")
axes[1].set_title("Residual (original − reconstruction)")
axes[1].legend(fontsize=9)

fig.suptitle("Section 2 — Reconstruction Quality", fontweight="bold")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "02_reconstruction.png", dpi=120, bbox_inches="tight")
plt.show()

# (c) Component spectra
plot_component_spectra(
    ssd_comps, fs=FS, nperseg=256,
    save_path=str(RESULTS_DIR / "02_component_spectra.png"),
)
print("Saved: 02_ssd_decomp.png, 02_reconstruction.png, 02_component_spectra.png")

Saved: 02_ssd_decomp.png, 02_reconstruction.png, 02_component_spectra.png


## Section 3 — Randomised SSD (rSSD)

**References:**
- Halko, Martinsson & Tropp (2011). *Finding Structure with Randomness: Probabilistic
  Algorithms for Constructing Approximate Matrix Decompositions.* SIAM Review.
- Kotala et al. (2025). *Randomized SSD.* BMSC.

Randomised SVD (rSVD) replaces the exact SVD with a rank-`k` approximation obtained via
a random projection. The sketch requires only `k + p` matrix-vector products (where `p`
is oversampling), giving **O(mn·k)** complexity vs **O(mn·min(m,n))** for the full SVD.

Two entry points in this codebase:
- `rsvd(X, k, n_oversamples, n_power_iter)` — standalone randomised SVD
- `svd_decompose(X, rank=k, method="randomized")` — rSVD inside the SSA framework
- `IncrementalSSD(use_rsvd=True)` — full SSD engine with rSVD backend

In [74]:
# ── 3.1  rSVD at k = 5, 10, 20 — singular values and reconstruction ─────────

L_s3 = 100   # embedding length for the Hankel matrix in this section
X_s3 = build_trajectory_matrix(signal, L=L_s3)
U_full_s3, S_full_s3, Vt_full_s3 = np.linalg.svd(X_s3, full_matrices=False)

ks = [5, 10, 20]   # rank values to compare
rsvd_res = {}

for k in ks:
    t0 = time.perf_counter()
    U_r, S_r, Vt_r = rsvd(X_s3, k=k, n_oversamples=5, n_power_iter=2, seed=SEED)
    elapsed_us = 1e6 * (time.perf_counter() - t0)
    X_approx = U_r @ np.diag(S_r) @ Vt_r
    # Compare against exact rank-k truncation
    X_exact_k = U_full_s3[:, :k] @ np.diag(S_full_s3[:k]) @ Vt_full_s3[:k, :]
    rel_err = (np.linalg.norm(X_exact_k - X_approx, 'fro')
               / np.linalg.norm(X_s3, 'fro'))
    recon = diagonal_averaging(X_approx)
    rsvd_res[k] = dict(S=S_r, recon=recon[:N_SIGNAL], rel_err=rel_err, us=elapsed_us)
    print(f"  k={k:2d}: rel_err={rel_err:.6f}   time={elapsed_us:.0f} µs")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Singular value comparison
idx_all = np.arange(1, 21)
axes[0].bar(idx_all, S_full_s3[:20], color="steelblue", alpha=0.5, label="full SVD")
for k, style in zip(ks, ["o-", "s--", "^:"]):
    axes[0].plot(np.arange(1, k + 1), rsvd_res[k]["S"],
                 style, markersize=5, label=f"rSVD k={k}")
axes[0].set_title("Singular values: full SVD vs rSVD")
axes[0].set_xlabel("rank index")
axes[0].set_ylabel("singular value")
axes[0].legend(fontsize=8)

# Reconstruction comparison
axes[1].plot(t, signal, linewidth=0.6, color="gray", alpha=0.5, label="original")
for k, color in zip(ks, ["crimson", "darkorange", "green"]):
    r = rsvd_res[k]["recon"]
    axes[1].plot(t[:len(r)], r, linewidth=0.8, color=color,
                 alpha=0.9, label=f"rSVD k={k} (err={rsvd_res[k]['rel_err']:.4f})")
axes[1].set_title("Signal reconstructed from rank-k rSVD")
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("amplitude")
axes[1].legend(fontsize=8)

fig.suptitle("Section 3 — Randomised SVD: k comparison", fontweight="bold")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "03_rsvd_k_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

  k= 5: rel_err=0.279326   time=2248 µs
  k=10: rel_err=0.372391   time=1596 µs
  k=20: rel_err=0.310330   time=2875 µs


In [75]:
# ── 3.2  Runtime comparison: full SVD vs rSVD at multiple signal lengths ────

N_vals   = [500, 1000, 2000, 4000]   # signal lengths to benchmark
k_bench  = 10                         # fixed rank for rSVD
L_frac   = 0.15                       # L = 15 % of N
n_reps   = 5                          # repetitions for stable timing

timing_rows = []
for N_val in N_vals:
    sig_t = two_sinusoids(N=N_val, f1=20, f2=80, fs=FS, seed=SEED)
    L_t   = max(10, int(N_val * L_frac))
    X_t   = build_trajectory_matrix(sig_t, L=L_t)

    # Full SVD
    ts_full = []
    for _ in range(n_reps):
        t0 = time.perf_counter()
        np.linalg.svd(X_t, full_matrices=False)
        ts_full.append(time.perf_counter() - t0)

    # rSVD
    ts_rsvd = []
    for _ in range(n_reps):
        t0 = time.perf_counter()
        rsvd(X_t, k=k_bench, n_oversamples=5, n_power_iter=2, seed=SEED)
        ts_rsvd.append(time.perf_counter() - t0)

    t_full_ms = 1000 * np.median(ts_full)
    t_rsvd_ms = 1000 * np.median(ts_rsvd)
    speedup   = t_full_ms / t_rsvd_ms

    # Approximation error
    Uf, Sf, Vft = np.linalg.svd(X_t, full_matrices=False)
    Ur, Sr, Vrt = rsvd(X_t, k=k_bench, n_oversamples=5, n_power_iter=2, seed=SEED)
    X_full_k = Uf[:, :k_bench] @ np.diag(Sf[:k_bench]) @ Vft[:k_bench, :]
    X_rsvd_k = Ur @ np.diag(Sr) @ Vrt
    rel_err  = (np.linalg.norm(X_full_k - X_rsvd_k, 'fro')
                / np.linalg.norm(X_t, 'fro'))

    timing_rows.append(dict(
        N=N_val, L=L_t, shape=f"{L_t}×{X_t.shape[1]}",
        full_svd_ms=round(t_full_ms, 2),
        rsvd_ms=round(t_rsvd_ms, 2),
        speedup=round(speedup, 2),
        rel_error=round(rel_err, 6),
    ))

timing_df = pd.DataFrame(timing_rows)
print(f"Runtime comparison (k={k_bench}, median of {n_reps} reps):")
print(timing_df.to_string(index=False))

# Plot speedup vs N
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(timing_df["N"], timing_df["speedup"], "o-", color="purple", markersize=7)
ax.axhline(1, linestyle="--", color="gray", linewidth=1, label="break-even")
ax.set_title(f"rSVD speedup over full SVD  (rank k={k_bench})")
ax.set_xlabel("signal length N")
ax.set_ylabel("speedup  (×)")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "03_rsvd_speedup.png", dpi=120, bbox_inches="tight")
plt.show()

Runtime comparison (k=10, median of 5 reps):
   N   L    shape  full_svd_ms  rsvd_ms  speedup  rel_error
 500  75   75×426         1.33     0.24     5.49        0.0
1000 150  150×851         8.08     0.50    16.14        0.0
2000 300 300×1701        27.42     1.15    23.93        0.0
4000 600 600×3401       135.11     2.98    45.34        0.0


In [76]:
# ── 3.3  IncrementalSSD with rSVD backend vs baseline SSD ──────────────────

w_s3 = signal[:500]   # single window for engine comparison

for use_r, label in [(False, "SSD (full SVD)"), (True, "IncrementalSSD (rSVD)")]:
    if use_r:
        eng = IncrementalSSD(fs=FS, use_rsvd=True,
                             rsvd_oversamples=5, rsvd_power_iter=2,
                             nmse_threshold=NMSE_THR, max_iter=MAX_ITER)
    else:
        eng = SSD(fs=FS, nmse_threshold=NMSE_THR, max_iter=MAX_ITER)
    t0 = time.perf_counter()
    c  = eng.fit(w_s3)
    elapsed_ms = 1000 * (time.perf_counter() - t0)
    r  = np.sum(c[:-1], axis=0)
    print(f"  {label:30s}: {len(c)-1:2d} comps  "
          f"NMSE={nmse(w_s3 - r, w_s3):.5f}  t={elapsed_ms:.1f} ms")

  SSD (full SVD)                :  5 comps  NMSE=0.00683  t=47.3 ms
  IncrementalSSD (rSVD)         :  5 comps  NMSE=0.00683  t=56.3 ms


## Section 4 — Sliding-Window Decomposition (Streaming SSD, Naive)

**Reference:** Harmouche, Fourer, Auger, Borgnat & Flandrin (2017). *The Sliding Singular
Spectrum Analysis: A Data-Driven Nonstationary Signal Decomposition Tool.*
IEEE Transactions on Signal Processing.

We run the `WindowManager` + `SSD.fit` loop **without any component matching**. Each
window produces an independent set of components that may be reordered relative to the
previous window. This section visualises the **identity-jumping problem** that motivates
Section 5.

In [77]:
# ── 4.1  Naive sliding-window loop — collect raw per-window components ──────

wm_naive  = WindowManager(window_len=WINDOW_LEN, stride=STRIDE, fs=FS)
eng_naive = SSD(fs=FS, nmse_threshold=NMSE_THR, max_iter=MAX_ITER)

raw_windows = []   # list of dicts: {start, components, dominant_freqs}

for sample_idx, x in enumerate(signal):
    win = wm_naive.push(float(x))
    if win is None:
        continue
    comps = eng_naive.fit(win)[:-1]   # drop residual
    freqs = [dominant_frequency(c, fs=FS) for c in comps]
    raw_windows.append(dict(
        start=sample_idx - WINDOW_LEN + 1,
        comps=comps,
        freqs=freqs,
    ))

print(f"Processed {len(raw_windows)} windows  "
      f"(window_len={WINDOW_LEN}, stride={STRIDE}, "
      f"overlap={wm_naive.overlap})")
print(f"Components per window: min={min(len(w['comps']) for w in raw_windows)}  "
      f"max={max(len(w['comps']) for w in raw_windows)}")

Processed 19 windows  (window_len=300, stride=150, overlap=150)
Components per window: min=2  max=6


In [78]:
# ── 4.2  Visualise identity jumping (without matching) ──────────────────────

MAX_SHOW = 4   # plot up to this many component slots (padded with NaN)

# Build per-slot frequency matrix  [n_windows x MAX_SHOW]
freq_matrix = np.full((len(raw_windows), MAX_SHOW), np.nan)
for wi, wd in enumerate(raw_windows):
    for ci, f in enumerate(wd["freqs"][:MAX_SHOW]):
        freq_matrix[wi, ci] = f

window_centers = np.array([w["start"] + WINDOW_LEN // 2 for w in raw_windows]) / FS

fig, axes = plt.subplots(2, 1, figsize=(13, 7))

# Panel A — dominant frequency per slot (no matching → jumpy)
for slot in range(MAX_SHOW):
    valid = ~np.isnan(freq_matrix[:, slot])
    axes[0].plot(window_centers[valid], freq_matrix[valid, slot],
                 "o", markersize=4, label=f"slot {slot}")
axes[0].set_title("Naive sliding SSD — dominant frequency per output slot\n"
                  "(no matching: slots reassigned each window → apparent jumps)")
axes[0].set_xlabel("window centre time (s)")
axes[0].set_ylabel("dominant frequency (Hz)")
axes[0].legend(fontsize=8)

# Panel B — first 5 windows stacked
n_disp = min(5, len(raw_windows))
offset = 0.0
for wi in range(n_disp):
    wd = raw_windows[wi]
    xs = np.arange(len(wd["comps"][0])) / FS + wd["start"] / FS
    for ci, comp in enumerate(wd["comps"][:MAX_SHOW]):
        axes[1].plot(xs, comp + offset, linewidth=0.7,
                     color=PALETTE[ci % len(PALETTE)])
    offset += 3.0
axes[1].set_title(f"First {n_disp} windows: raw components (colour = output slot, not component identity)")
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("amplitude (offset)")

fig.suptitle("Section 4 — Naive Sliding Window (no matching)", fontweight="bold")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "04_naive_sliding.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: 04_naive_sliding.png")

Saved: 04_naive_sliding.png


## Section 5 — Component Matching & Temporal Consistency

The Hungarian algorithm (Kuhn–Munkres, via `scipy.optimize.linear_sum_assignment`) solves
the minimum-cost bipartite matching between components in consecutive windows. The
`ComponentMatcher` class implements three cost metrics:

| Metric | Formula | Reference |
|--------|---------|----------|
| `d_corr` | `1 − |⟨x,y⟩| / (‖x‖·‖y‖)` | Harmouche et al. 2017 |
| `d_freq` | `|f_max(x) − f_max(y)|` | spectral centroid |
| `hybrid` | `(1−w)·d_corr + w·d_freq_norm` | this work |

Parameters `max_cost` (rejection threshold), `max_trajectories` (id cap), and `lookback`
(active-trajectory window) control robustness.

In [79]:
# ── 5.1  Compare all three distance modes on two consecutive windows ─────────

ssd_w = SSD(fs=FS, nmse_threshold=NMSE_THR, max_iter=MAX_ITER)
w_prev = signal[:WINDOW_LEN]
w_curr = signal[STRIDE:STRIDE + WINDOW_LEN]
c_prev = ssd_w.fit(w_prev)[:-1]
c_curr = ssd_w.fit(w_curr)[:-1]
overlap = wm_naive.overlap

print(f"prev: {len(c_prev)} comps  |  curr: {len(c_curr)} comps  |  overlap: {overlap}")

for dist in ["d_corr", "d_freq", "hybrid"]:
    m = ComponentMatcher(distance=dist, freq_weight=0.5, fs=FS)
    cost = m.build_cost_matrix(c_prev, c_curr, overlap)
    mapping = m.match(c_prev, c_curr, overlap)
    print(f"  {dist:8s}: cost_matrix shape={cost.shape}  mapping={mapping}")

# Visualise the matching graph (d_freq mode)
m_vis  = ComponentMatcher(distance="d_freq", freq_weight=1.0, fs=FS)
C_vis  = m_vis.build_cost_matrix(c_prev, c_curr, overlap)
map_vis = m_vis.match(c_prev, c_curr, overlap)
plot_matching_graph(
    c_prev, c_curr, map_vis, overlap=overlap, cost_matrix=C_vis,
    save_path=str(RESULTS_DIR / "05_matching_graph.png"),
)
print("Saved: 05_matching_graph.png")

prev: 5 comps  |  curr: 5 comps  |  overlap: 150
  d_corr  : cost_matrix shape=(5, 5)  mapping={0: 3, 1: 1, 2: 2, 3: 0, 4: 4}
  d_freq  : cost_matrix shape=(5, 5)  mapping={0: 0, 1: 1, 2: 4, 3: 2, 4: 3}
  hybrid  : cost_matrix shape=(5, 5)  mapping={0: 3, 1: 1, 2: 2, 3: 0, 4: 4}
Saved: 05_matching_graph.png


In [80]:
# ── 5.2  Stateful API: matched vs unmatched trajectory plots ─────────────────

def run_matched_pipeline(distance, freq_weight=0.5, max_cost=0.5,
                          max_trajectories=None):
    """Run full streaming loop with a given matcher config, return trajectory store."""
    wm = WindowManager(window_len=WINDOW_LEN, stride=STRIDE, fs=FS)
    eng = SSD(fs=FS, nmse_threshold=NMSE_THR, max_iter=MAX_ITER)
    matcher = ComponentMatcher(
        distance=distance, freq_weight=freq_weight, fs=FS,
        lookback=3, max_cost=max_cost,
        max_trajectories=max_trajectories,
    )
    store = TrajectoryStore(max_components=MAX_COMP, max_len=N_SIGNAL)
    for i, x in enumerate(signal):
        w = wm.push(float(x))
        if w is None:
            continue
        comps = eng.fit(w)[:-1]
        mapping = matcher.match_stateful(comps, wm.overlap)
        store.update(i - WINDOW_LEN + 1, comps, mapping, wm.overlap)
    return store

stores = {}
for dist in ["d_corr", "d_freq", "hybrid"]:
    stores[dist] = run_matched_pipeline(
        distance=dist, freq_weight=0.5, max_cost=0.5, max_trajectories=6
    )
    n_traj = len(stores[dist].get_all())
    print(f"  {dist:8s}: {n_traj} distinct trajectories")

  d_corr  : 6 distinct trajectories
  d_freq  : 6 distinct trajectories
  hybrid  : 6 distinct trajectories


In [81]:
# ── 5.3  Colour-coded trajectory plots: matched vs unmatched ────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Top row: d_corr and d_freq matched trajectories
for ax, dist in zip(axes[0], ["d_corr", "d_freq"]):
    ax.plot(t, signal, linewidth=0.5, color="gray", alpha=0.3, label="signal")
    for tid, arr in stores[dist].get_all().items():
        k = min(len(arr), N_SIGNAL)
        tx = np.arange(k) / FS
        ax.plot(tx, arr[:k], linewidth=1.0,
                color=PALETTE[tid % len(PALETTE)], label=f"traj {tid}")
    ax.set_title(f"Matched trajectories ({dist})")
    ax.set_xlabel("time (s)"); ax.set_ylabel("amplitude")
    ax.legend(fontsize=7, loc="upper right")

# Bottom-left: hybrid
ax_h = axes[1, 0]
ax_h.plot(t, signal, linewidth=0.5, color="gray", alpha=0.3)
for tid, arr in stores["hybrid"].get_all().items():
    k = min(len(arr), N_SIGNAL)
    ax_h.plot(np.arange(k) / FS, arr[:k], linewidth=1.0,
              color=PALETTE[tid % len(PALETTE)], label=f"traj {tid}")
ax_h.set_title("Matched trajectories (hybrid)")
ax_h.set_xlabel("time (s)"); ax_h.set_ylabel("amplitude")
ax_h.legend(fontsize=7, loc="upper right")

# Bottom-right: naive (no matching) — slot 0 of each window as scatter
ax_n = axes[1, 1]
ax_n.plot(t, signal, linewidth=0.5, color="gray", alpha=0.3, label="signal")
for slot in range(min(4, MAX_SHOW)):
    xs_plot, ys_plot = [], []
    for wd in raw_windows:
        if slot < len(wd["comps"]):
            c = wd["comps"][slot]
            xs_plot.extend(np.arange(len(c)) / FS + wd["start"] / FS)
            ys_plot.extend(c.tolist())
    ax_n.plot(xs_plot, ys_plot, linewidth=0.6,
              color=PALETTE[slot % len(PALETTE)], alpha=0.7, label=f"slot {slot}")
ax_n.set_title("Naive (no matching) — colour = output slot, not identity")
ax_n.set_xlabel("time (s)"); ax_n.set_ylabel("amplitude")
ax_n.legend(fontsize=7, loc="upper right")

fig.suptitle("Section 5 — Matched vs Unmatched Trajectories", fontweight="bold")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "05_traj_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: 05_traj_comparison.png")

Saved: 05_traj_comparison.png


In [82]:
# ── 5.4  max_cost rejection, max_trajectories cap, lookback ────────────────

# max_cost rejection: tight threshold forces new IDs
m_tight = ComponentMatcher(distance="d_freq", freq_weight=1.0, fs=FS,
                            max_cost=0.01, max_trajectories=20)
m_tight.match_stateful(c_prev, overlap)
m_t2 = m_tight.match_stateful(c_curr, overlap)
print(f"max_cost=0.01 (tight): {m_t2}")

# max_trajectories cap: hard cap at 2
m_cap = ComponentMatcher(distance="d_freq", freq_weight=1.0, fs=FS,
                          max_cost=0.9, max_trajectories=2)
m_c0 = m_cap.match_stateful(c_prev, overlap)
n_dropped = sum(1 for v in m_c0.values() if v == ComponentMatcher.DROP_ID)
print(f"max_trajectories=2: {m_c0}  ({n_dropped} component(s) dropped with id=-1)")

# lookback: feed 5 windows and print progression
m_lb = ComponentMatcher(distance="d_freq", freq_weight=1.0, fs=FS,
                         lookback=5, max_cost=0.8, max_trajectories=10)
eng_lb = SSD(fs=FS, nmse_threshold=NMSE_THR, max_iter=MAX_ITER)
for wi in range(5):
    start, end = wi * STRIDE, wi * STRIDE + WINDOW_LEN
    if end > N_SIGNAL:
        break
    c_lb = eng_lb.fit(signal[start:end])[:-1]
    m_lb_res = m_lb.match_stateful(c_lb, overlap)
    print(f"  lookback window {wi}: {m_lb_res}")

max_cost=0.01 (tight): {0: 0, 1: 1, 2: 5, 3: 6, 4: 7}
max_trajectories=2: {0: 0, 1: 1, 2: -1, 3: -1, 4: -1}  (3 component(s) dropped with id=-1)
  lookback window 0: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}
  lookback window 1: {0: 0, 1: 1, 2: 4, 3: 2, 4: 3}
  lookback window 2: {0: 0, 1: 1, 2: 2, 3: 4, 4: 3, 5: 5}
  lookback window 3: {0: 0, 1: 1, 2: 3, 3: 4, 4: 2, 5: 5}
  lookback window 4: {0: 0, 1: 1, 2: 3}


## Section 6 — Incremental SVD Update (O(k²) Sliding Update)

**References:**
- Brand (2003). *Fast online SVD revisions for lightweight recommender systems.* SDM.
- Saeed, Cheong Took & Alty (2020). *An Efficient Algorithm to Implement the Sliding
  Singular Spectrum Analysis.* ICASSP.
- Balzano & Wright (2013). *On GROUSE and Incremental SVD.* arXiv.

The `RankOneUpdater` maintains a rank-`r` factorisation `X ≈ U S Vᵀ`. When the
Hankel window slides by one sample, the matrix changes by two outer-product terms:
one to remove the departing column/row, one to add the arriving one. Each update costs
O(r²·L) vs O(L·K·min(L,K)) for a full rebuild.

In [83]:
# ── 6.1  RankOneUpdater: rank-1 update and slide_window ─────────────────────

L_s6  = 50    # embedding dimension for this section
rank  = 6     # number of singular vectors to maintain
x_s6  = signal[:600]   # 600-sample excerpt

X_h6  = _build_hankel(x_s6[:500], L_s6)
U0, S0, Vt0 = svd_decompose(X_h6, rank=rank)
updater = RankOneUpdater(U0, S0, Vt0, refresh_every=200)
print(f"RankOneUpdater initialised: U{updater.U.shape} S{updater.S.shape} Vt{updater.Vt.shape}")

# Single rank-1 update: X_new = X_old + a·bᵀ
rng_s6 = np.random.default_rng(SEED)
a_vec  = rng_s6.standard_normal(L_s6) * 0.01
b_vec  = rng_s6.standard_normal(X_h6.shape[1]) * 0.01
U_upd, S_upd, Vt_upd = updater.update(a_vec, b_vec)

X_exact = X_h6 + np.outer(a_vec, b_vec)
X_approx = U_upd @ np.diag(S_upd) @ Vt_upd
rel_update_err = (np.linalg.norm(X_exact - X_approx, 'fro')
                   / np.linalg.norm(X_exact, 'fro'))
print(f"Rank-1 update: relative Frobenius error = {rel_update_err:.6f}")

# Slide window 20 steps and track singular values
sv_history = [updater.S.copy()]
for step in range(20):
    new_sample = float(x_s6[500 + step])
    window_data = x_s6[step: step + 500]
    updater.slide_window(new_sample, window_data)
    sv_history.append(updater.S.copy())

print(f"After 20 slide steps — singular values: {updater.S.round(2)}")

RankOneUpdater initialised: U(50, 6) S(6,) Vt(6, 451)
Rank-1 update: relative Frobenius error = 0.106801
After 20 slide steps — singular values: [78.98 75.13 64.39 62.18 35.67 24.48]


In [84]:
# ── 6.2  Singular values tracked over time ──────────────────────────────────

sv_arr = np.array(sv_history)   # shape (n_steps+1, rank)

fig, ax = plt.subplots(figsize=(10, 4))
for r_idx in range(rank):
    ax.plot(sv_arr[:, r_idx], "-o", markersize=3,
            color=PALETTE[r_idx % len(PALETTE)],
            label=f"σ_{r_idx + 1}")
ax.set_title("Singular values tracked via incremental slide_window updates")
ax.set_xlabel("slide step")
ax.set_ylabel("singular value")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "06_sv_tracking.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: 06_sv_tracking.png")

Saved: 06_sv_tracking.png


In [85]:
# ── 6.3  Numerical accuracy: incremental vs full SVD rebuild ─────────────────

n_slides    = 30
residual_norms = []

# Fresh updater
X_acc = _build_hankel(x_s6[:500], L_s6)
Uf, Sf, Vtf = svd_decompose(X_acc, rank=rank)
upd_acc = RankOneUpdater(Uf, Sf, Vtf, refresh_every=999)

for step in range(n_slides):
    new_s = float(x_s6[500 + step])
    win_data = x_s6[step: step + 500]
    upd_acc.slide_window(new_s, win_data)

    # Ground truth: full SVD of the true slid Hankel
    X_true = _build_hankel(x_s6[step + 1: step + 501], L_s6)
    U_true, S_true, _ = svd_decompose(X_true, rank=rank)

    # Compare subspace (use principal angle)
    theta = subspace_angle(U_true, upd_acc.U)
    sv_diff = np.linalg.norm(S_true - upd_acc.S)
    residual_norms.append(dict(step=step, subspace_angle_rad=theta, sv_diff=sv_diff))

acc_df = pd.DataFrame(residual_norms)
print(acc_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(acc_df["step"], acc_df["subspace_angle_rad"], "o-", markersize=4, color="steelblue")
axes[0].set_title("Principal angle: incremental U vs full-SVD U")
axes[0].set_xlabel("slide step"); axes[0].set_ylabel("angle (rad)")

axes[1].plot(acc_df["step"], acc_df["sv_diff"], "s-", markersize=4, color="darkorange")
axes[1].set_title("||S_incremental − S_fullSVD||₂")
axes[1].set_xlabel("slide step"); axes[1].set_ylabel("norm diff")

fig.suptitle("Section 6 — Incremental SVD Accuracy", fontweight="bold")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "06_incremental_accuracy.png", dpi=120, bbox_inches="tight")
plt.show()

 step  subspace_angle_rad  sv_diff
    0        4.214685e-08 0.630990
    1        6.122891e-06 1.378398
    2        1.526246e-05 2.073410
    3        3.529718e-05 2.603546
    4        1.557137e-04 2.900168
    5        3.495596e-04 2.942315
    6        5.280173e-04 2.776628
    7        6.117874e-04 2.552613
    8        6.785224e-04 2.535692
    9        7.573109e-04 2.751236
   10        7.744962e-04 2.879306
   11        8.538877e-04 3.086910
   12        9.544230e-04 3.535847
   13        1.009056e-03 4.067795
   14        1.051096e-03 4.516607
   15        9.677997e-04 4.814209
   16        8.227141e-04 4.979708
   17        5.824820e-04 5.093759
   18        6.890804e-04 5.236143
   19        1.221720e-03 5.456434
   20        1.929426e-03 5.744061
   21        2.622597e-03 6.072186
   22        3.404017e-03 6.429483
   23        4.105576e-03 6.768394
   24        4.523658e-03 6.983738
   25        4.056430e-03 7.035998
   26        3.443770e-03 6.968602
   27        2.84241

In [86]:
# ── 6.4  Runtime: full SVD rebuild vs slide_window per step ─────────────────

n_timing_steps = 50
t_full_rebuild, t_incremental = [], []

# Build a fresh updater for timing
X_tim = _build_hankel(x_s6[:500], L_s6)
Uf_t, Sf_t, Vtf_t = svd_decompose(X_tim, rank=rank)
upd_tim = RankOneUpdater(Uf_t, Sf_t, Vtf_t, refresh_every=9999)

for step in range(n_timing_steps):
    new_s = float(x_s6[min(500 + step, len(x_s6) - 1)])
    win_data = x_s6[step: step + 500]

    # Full rebuild timing
    t0 = time.perf_counter()
    X_rebuild = _build_hankel(win_data, L_s6)
    svd_decompose(X_rebuild, rank=rank)
    t_full_rebuild.append(1000 * (time.perf_counter() - t0))

    # Incremental timing
    t0 = time.perf_counter()
    upd_tim.slide_window(new_s, win_data)
    t_incremental.append(1000 * (time.perf_counter() - t0))

print(f"Full SVD rebuild  — median: {np.median(t_full_rebuild):.3f} ms/step")
print(f"slide_window incr — median: {np.median(t_incremental):.3f} ms/step")
print(f"Speedup: {np.median(t_full_rebuild)/np.median(t_incremental):.1f}×")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t_full_rebuild, linewidth=0.8, label=f"full SVD rebuild (rank={rank})")
ax.plot(t_incremental,  linewidth=0.8, label="slide_window incremental")
ax.set_title("Runtime per step: full SVD rebuild vs incremental update")
ax.set_xlabel("step"); ax.set_ylabel("time (ms)")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "06_timing_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

Full SVD rebuild  — median: 0.771 ms/step
slide_window incr — median: 0.138 ms/step
Speedup: 5.6×


## Section 7 — Full Streaming Pipeline (End-to-End)

This section combines all components into a single streaming loop:

```
sample generator  →  WindowManager  →  SSD.fit  →
ComponentMatcher.match_stateful  →  TrajectoryStore.update
```

The signal is fed **sample-by-sample** via a Python generator — the pipeline never
looks ahead. All stability metrics are computed per window. The final trajectory store
is compared against the offline SSD baseline from Section 2.

In [87]:
# ── 7.1  Sample-by-sample generator (causal — no look-ahead) ────────────────

def sample_stream(sig: np.ndarray):
    """Yield one scalar sample at a time — simulates a live ADC feed."""
    for x in sig:
        yield float(x)

# Confirm it is truly lazy
gen = sample_stream(signal)
first_five = [next(gen) for _ in range(5)]
print("First 5 samples from stream:", [round(v, 4) for v in first_five])

First 5 samples from stream: [1.0306, 1.2027, 1.6551, 1.8854, 1.7233]


In [88]:
# ── 7.2  Full streaming loop ────────────────────────────────────────────────

wm7      = WindowManager(window_len=WINDOW_LEN, stride=STRIDE, fs=FS)
eng7     = SSD(fs=FS, nmse_threshold=NMSE_THR, max_iter=MAX_ITER)
matcher7 = ComponentMatcher(
    distance="d_freq", freq_weight=1.0, fs=FS,
    lookback=1000, max_cost=0.10, max_trajectories=MAX_COMP,
)
store7   = TrajectoryStore(max_components=MAX_COMP, max_len=N_SIGNAL)

metrics_rows7  = []
pipeline_recs7 = []
prev_comps7    = None
prev_S7        = None
win_idx7       = 0

for sample_idx, x in enumerate(sample_stream(signal)):
    window = wm7.push(x)
    if window is None:
        continue

    comps = eng7.fit(window)[:-1]
    mapping = matcher7.match_stateful(comps, wm7.overlap)
    prev_win_map = matcher7.previous_window_mapping()

    win_start = sample_idx - WINDOW_LEN + 1
    store7.update(win_start, comps, mapping, wm7.overlap)

    recon7 = np.sum(comps, axis=0) if comps else np.zeros_like(window)
    q7     = qrf(window, recon7)

    Lw7 = max(2, WINDOW_LEN // 3)
    Xw7 = build_trajectory_matrix(window, Lw7)
    _, S7_curr, _ = svd_decompose(Xw7)
    sv_d = singular_value_drift(S7_curr, prev_S7)
    prev_S7 = S7_curr

    ec7 = energy_continuity(comps, prev_comps7, prev_win_map)

    if prev_comps7 is not None and prev_win_map:
        C7  = matcher7.build_cost_matrix(prev_comps7, comps, wm7.overlap)
        mc7 = matching_confidence(C7, prev_win_map)
    else:
        mc7 = float("nan")

    fmax_row = {}
    for ci, comp in enumerate(comps):
        traj_id = mapping.get(ci, -1)
        if traj_id >= 0:
            fmax_row[f"f_max_t{traj_id}"] = dominant_frequency(comp, fs=FS)

    row = {"window_index": win_idx7, "qrf": q7,
           "singular_value_drift": sv_d,
           "energy_continuity": ec7,
           "matching_confidence": mc7}
    row.update(fmax_row)
    metrics_rows7.append(row)

    pipeline_recs7.append(dict(
        window_idx=win_idx7, sample_start=win_start,
        window_signal=window.copy(),
        components=[c.copy() for c in comps],
    ))

    prev_comps7 = comps
    win_idx7 += 1

metrics7_df   = pd.DataFrame(metrics_rows7)
metrics7_path = str(RESULTS_DIR / "metrics.csv")
metrics7_df.to_csv(metrics7_path, index=False)

print(f"Processed {win_idx7} windows.")
print(metrics7_df[["window_index", "qrf", "singular_value_drift",
                    "matching_confidence"]].head())


Processed 19 windows.
   window_index        qrf  singular_value_drift  matching_confidence
0             0  20.351939                   NaN                  NaN
1             1  21.611801              3.747692             0.986667
2             2  23.481225              3.784173             1.000000
3             3  21.932896              5.683492             0.993333
4             4  20.496243             39.196116             0.995556


In [89]:
# ── 7.3  Online reconstruction vs offline SSD baseline ──────────────────────

trajs7 = store7.get_all()
full_recon7 = np.zeros(N_SIGNAL, dtype=np.float64)
coverage7   = np.zeros(N_SIGNAL, dtype=np.float64)
for arr in trajs7.values():
    k = min(len(arr), N_SIGNAL)
    valid = ~np.isnan(arr[:k])
    full_recon7[:k][valid] += arr[:k][valid]
    coverage7[:k][valid]   += 1.0

covered7 = coverage7 > 0
online_recon_masked = np.where(covered7, full_recon7, np.nan)

global_qrf7  = qrf(signal[covered7], full_recon7[covered7])
global_nmse7 = nmse(signal[covered7] - full_recon7[covered7], signal[covered7])
print(f"Online  — Global QRF: {global_qrf7:.2f} dB   NMSE: {global_nmse7:.6f}")
print(f"Offline — QRF: {s2_results['B-nonstat']['qrf']:.2f} dB   "
      f"NMSE: {s2_results['B-nonstat']['nmse']:.6f}")

fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)

axes[0].plot(t, signal, linewidth=0.6, color="steelblue", label="original")
axes[0].plot(t, online_recon_masked, linewidth=0.8, color="darkorange",
             alpha=0.85, label=f"online stream (QRF={global_qrf7:.1f} dB)")
axes[0].plot(t, ssd_recon, linewidth=0.8, color="green",
             alpha=0.7, linestyle="--",
             label=f"offline SSD (QRF={s2_results['B-nonstat']['qrf']:.1f} dB)")
axes[0].set_ylabel("amplitude")
axes[0].set_title("Online reconstruction vs offline SSD baseline")
axes[0].legend(fontsize=8)

axes[1].plot(t, signal - online_recon_masked, linewidth=0.6,
             color="crimson", label="online residual")
axes[1].set_ylabel("amplitude")
axes[1].set_title("Online residual")
axes[1].legend(fontsize=8)

# Component trajectories heatmap
traj_matrix = np.full((len(trajs7), N_SIGNAL), np.nan)
for row_i, (tid, arr) in enumerate(sorted(trajs7.items())):
    k = min(len(arr), N_SIGNAL)
    traj_matrix[row_i, :k] = arr[:k]

axes[2].imshow(traj_matrix, aspect="auto", cmap="RdBu_r",
               vmin=-2, vmax=2,
               extent=[0, N_SIGNAL / FS, len(trajs7) - 0.5, -0.5])
axes[2].set_title("Component trajectory heatmap (row = trajectory ID)")
axes[2].set_xlabel("time (s)")
axes[2].set_ylabel("trajectory ID")

fig.suptitle("Section 7 — Full Streaming Pipeline", fontweight="bold")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "07_streaming_pipeline.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: 07_streaming_pipeline.png")

Online  — Global QRF: 21.52 dB   NMSE: 0.007044
Offline — QRF: 20.77 dB   NMSE: 0.008378
Saved: 07_streaming_pipeline.png


In [90]:
# ── 7.4  All visualization helpers ─────────────────────────────────────────

# Save run_summary.json (needed by plot_metrics)
summary7 = {}
traj_cols7 = [c for c in metrics7_df.columns if c.startswith("f_max_t")]
for col in traj_cols7:
    tid = int(col[len("f_max_t"):])
    v = freq_drift_aggregate(metrics7_df[col].values)
    summary7[f"freq_drift_t{tid}"] = None if not np.isfinite(v) else float(v)
with open(RESULTS_DIR / "run_summary.json", "w") as fh:
    json.dump(summary7, fh, indent=2)

# plot_trajectory_overlay
plot_trajectory_overlay(store7, signal, fs=FS,
    save_path=str(RESULTS_DIR / "07_traj_overlay.png"))

# plot_metrics_over_windows
plot_metrics_over_windows(metrics7_path,
    save_path=str(RESULTS_DIR / "07_metrics_over_windows.png"))

# plot_pipeline_dashboard
plot_pipeline_dashboard(signal, ssd_comps, store7, metrics7_path, fs=FS,
    save_path=str(RESULTS_DIR / "07_pipeline_dashboard.png"))

# plot_window_reconstruction — first, middle, last windows
for label, idx in [("first", 0), ("mid", len(pipeline_recs7)//2), ("last", -1)]:
    rec = pipeline_recs7[idx]
    plot_window_reconstruction(
        rec["window_signal"], rec["components"],
        rec["window_idx"], rec["sample_start"], fs=FS,
        save_path=str(RESULTS_DIR / f"07_win_{label}.png"),
    )

# plot_window_grid
plot_window_grid(pipeline_recs7, n_windows=9, fs=FS,
    save_path=str(RESULTS_DIR / "07_window_grid.png"))

# plot_nmse_over_time
plot_nmse_over_time(signal, online_recon_masked, fs=FS,
    save_path=str(RESULTS_DIR / "07_nmse_over_time.png"))

# plot_metrics (advanced 2-panel)
plot_metrics(RESULTS_DIR, show=False)

print("All visualization functions called and saved.")


All visualization functions called and saved.


## Section 8 — Quantitative Evaluation

Systematic benchmark across all five engines, three signal types, and three SNR levels.
Metrics reported:
- **NMSE** — normalised MSE of per-window reconstruction
- **QRF** — quality-of-reconstruction factor (dB) (Harmouche 2017)
- **Wall-clock time** (ms/window)
- **Peak memory** (MB, via `tracemalloc`)

In [91]:
# ── 8.1  Benchmark: all engines × all signals × window ──────────────────────

BENCH_WLEN   = 300    # window length for benchmark [samples]
BENCH_STRIDE = 150    # stride [samples]
BENCH_N_WIN  = 8      # number of windows to process per configuration

bench_signals = {
    "stationary":     sig_A,
    "non-stationary": sig_B,
    "noisy":          sig_C,
}

def make_engine(name):
    """Instantiate engine by name with benchmark-friendly defaults."""
    if name == "SSD":
        return SSD(fs=FS, nmse_threshold=NMSE_THR, max_iter=10)
    elif name == "IncrementalSSD":
        return IncrementalSSD(fs=FS, nmse_threshold=NMSE_THR, max_iter=10)
    elif name == "IncrementalSSD-rSVD":
        return IncrementalSSD(fs=FS, use_rsvd=True, nmse_threshold=NMSE_THR, max_iter=10)
    elif name == "OptimizedSSD-fwhm":
        return OptimizedSSD(fs=FS, spectral_method="fwhm", nmse_threshold=NMSE_THR, max_iter=10)
    elif name == "OptimizedSSD-moment":
        return OptimizedSSD(fs=FS, spectral_method="moment", nmse_threshold=NMSE_THR, max_iter=10)
    elif name == "OptimizedSSD-gaussian":
        return OptimizedSSD(fs=FS, spectral_method="gaussian", nmse_threshold=NMSE_THR, max_iter=10)
    elif name == "SSA":
        return SSA(fs=FS, n_components=3, window_length=BENCH_WLEN // 3)
    raise ValueError(name)

engine_names = [
    "SSD",
    "IncrementalSSD",
    "IncrementalSSD-rSVD",
    "OptimizedSSD-fwhm",
    "OptimizedSSD-moment",
    "OptimizedSSD-gaussian",
    "SSA",
]

bench_rows = []

for sig_name, sig_bench in bench_signals.items():
    # Extract BENCH_N_WIN windows
    windows_bench = []
    wm_bench = WindowManager(window_len=BENCH_WLEN, stride=BENCH_STRIDE, fs=FS)
    for x in sig_bench:
        w = wm_bench.push(float(x))
        if w is not None:
            windows_bench.append(w)
        if len(windows_bench) == BENCH_N_WIN:
            break

    for eng_name in engine_names:
        eng = make_engine(eng_name)
        nmse_vals, qrf_vals, times_ms = [], [], []
        peak_mb = 0.0

        tracemalloc.start()
        for win in windows_bench:
            t0 = time.perf_counter()
            comps = eng.fit(win)
            elapsed = time.perf_counter() - t0
            _, peak = tracemalloc.get_traced_memory()
            peak_mb = max(peak_mb, peak / 1e6)
            comps_only = comps[:-1]
            if comps_only:
                recon = np.sum(comps_only, axis=0)
                nmse_vals.append(nmse(win - recon, win))
                qrf_vals.append(qrf(win, recon))
            times_ms.append(1000 * elapsed)
        tracemalloc.stop()

        bench_rows.append(dict(
            engine=eng_name,
            signal=sig_name,
            nmse_median=round(float(np.median(nmse_vals)), 5) if nmse_vals else np.nan,
            qrf_median=round(float(np.median(qrf_vals)), 2)   if qrf_vals  else np.nan,
            time_ms_median=round(float(np.median(times_ms)), 1),
            peak_mem_mb=round(peak_mb, 2),
        ))

bench_df = pd.DataFrame(bench_rows)
print(bench_df.to_string(index=False))

               engine         signal  nmse_median  qrf_median  time_ms_median  peak_mem_mb
                  SSD     stationary      0.00000       79.92            22.1         4.96
       IncrementalSSD     stationary      0.00000       79.92            24.9         5.11
  IncrementalSSD-rSVD     stationary      0.00000       79.92            24.4         4.91
    OptimizedSSD-fwhm     stationary      0.00000       79.92             4.0         0.89
  OptimizedSSD-moment     stationary      0.00000       79.92             4.2         0.89
OptimizedSSD-gaussian     stationary      0.00000       79.92            14.6         4.09
                  SSA     stationary      0.00000         inf            46.6         1.36
                  SSD non-stationary      0.00685       21.64            83.1         8.00
       IncrementalSSD non-stationary      0.00685       21.64            84.6         8.05
  IncrementalSSD-rSVD non-stationary      0.00685       21.64            88.3         6.85

In [92]:
# ── 8.2  Bar charts: NMSE, QRF, runtime ─────────────────────────────────────

sig_types  = list(bench_signals.keys())
n_eng      = len(engine_names)
x_pos      = np.arange(n_eng)
bar_width  = 0.25
eng_labels = [e.replace("-", "\n") for e in engine_names]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric, label in [
    (axes[0], "nmse_median",   "Median NMSE (lower=better)"),
    (axes[1], "qrf_median",    "Median QRF (dB, higher=better)"),
    (axes[2], "time_ms_median","Median runtime (ms/window)"),
]:
    for si, sig_n in enumerate(sig_types):
        vals = bench_df[bench_df["signal"] == sig_n][metric].values
        offset = (si - 1) * bar_width
        ax.bar(x_pos + offset, vals, bar_width,
               label=sig_n, color=PALETTE[si % len(PALETTE)], alpha=0.85)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(eng_labels, fontsize=7)
    ax.set_title(label, fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel("engine")

fig.suptitle("Section 8 — Engine Benchmark Summary", fontweight="bold")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "08_benchmark.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: 08_benchmark.png")

Saved: 08_benchmark.png


In [93]:
# ── 8.3  SNR sweep: all engines on signal B ──────────────────────────────────

SNR_LEVELS = [5, 10, 15, 20, 30, 40]   # dB
snr_rows   = []

for snr in SNR_LEVELS:
    sig_noisy_snr = chirp_plus_sinusoid(
        N=N_SIGNAL, f_sin=50.0, f_start=10.0, f_end=150.0,
        fs=FS, snr_db=snr, seed=SEED,
    )
    win_snr = sig_noisy_snr[:WINDOW_LEN]   # single window for speed
    for eng_name in ["SSD", "OptimizedSSD-fwhm", "IncrementalSSD"]:
        eng = make_engine(eng_name)
        comps = eng.fit(win_snr)
        comps_only = comps[:-1]
        if comps_only:
            recon = np.sum(comps_only, axis=0)
            snr_rows.append(dict(
                snr_db=snr, engine=eng_name,
                nmse=round(nmse(win_snr - recon, win_snr), 5),
                qrf_db=round(qrf(win_snr, recon), 2),
                n_comps=len(comps_only),
            ))

snr_df = pd.DataFrame(snr_rows)
print(snr_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for eng_n, style in zip(["SSD", "OptimizedSSD-fwhm", "IncrementalSSD"],
                         ["-o", "--s", ":^"]):
    sub = snr_df[snr_df["engine"] == eng_n]
    axes[0].plot(sub["snr_db"], sub["nmse"],   style, markersize=6, label=eng_n)
    axes[1].plot(sub["snr_db"], sub["qrf_db"], style, markersize=6, label=eng_n)

axes[0].set_title("NMSE vs SNR"); axes[0].set_xlabel("SNR (dB)"); axes[0].set_ylabel("NMSE")
axes[0].legend(fontsize=8)
axes[1].set_title("QRF vs SNR"); axes[1].set_xlabel("SNR (dB)"); axes[1].set_ylabel("QRF (dB)")
axes[1].legend(fontsize=8)

fig.suptitle("Section 8 — SNR Sweep", fontweight="bold")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "08_snr_sweep.png", dpi=120, bbox_inches="tight")
plt.show()

 snr_db            engine    nmse  qrf_db  n_comps
      5               SSD 0.00601   22.21        6
      5 OptimizedSSD-fwhm 0.00601   22.21        6
      5    IncrementalSSD 0.00601   22.21        6
     10               SSD 0.00553   22.57        8
     10 OptimizedSSD-fwhm 0.00634   21.98        7
     10    IncrementalSSD 0.00553   22.57        8
     15               SSD 0.00978   20.10        5
     15 OptimizedSSD-fwhm 0.00762   21.18        5
     15    IncrementalSSD 0.00978   20.10        5
     20               SSD 0.00922   20.35        5
     20 OptimizedSSD-fwhm 0.00891   20.50        4
     20    IncrementalSSD 0.00922   20.35        5
     30               SSD 0.00685   21.64        3
     30 OptimizedSSD-fwhm 0.00402   23.95        3
     30    IncrementalSSD 0.00685   21.64        3
     40               SSD 0.00607   22.17        3
     40 OptimizedSSD-fwhm 0.00327   24.85        3
     40    IncrementalSSD 0.00607   22.17        3


## Section 9 — Real-World Application Demo

> **Note:** No real-world dataset is shipped with this repository. We therefore use
> the **Rössler attractor** x-component — a deterministic chaotic dynamical system
> whose broadband, non-stationary spectrum closely mimics real IoT / physiological
> sensor data. The system is governed by:
>
> ẋ = −y − z,  ẏ = x + αy,  ż = β + z(x − γ)
>
> with parameters α=0.2, β=0.2, γ=5.7 (classic chaotic regime).

We apply the full streaming pipeline and interpret the extracted components
in terms of the known Rössler spectral structure (fundamental + harmonics).

The `component_onset` generator is used to demonstrate **dynamic onset detection** —
tracking the moment a new spectral component appears mid-stream.

In [94]:
# ── 9.1  Rössler attractor: generate, visualise, spectrum ───────────────────

N_ROSS  = 4000
sig_ross = rossler(N=N_ROSS, dt=0.02, alpha=0.2, beta=0.2, gamma=3.5, seed=SEED)
t_ross   = np.arange(N_ROSS) * 0.02   # virtual time axis (dt=0.02)
fs_ross  = 1.0 / 0.02   # 50 Hz effective sampling rate

from scipy.signal import welch as _welch
freqs_r, psd_r = _welch(sig_ross, fs=fs_ross, nperseg=256)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(t_ross, sig_ross, linewidth=0.7, color="purple")
axes[0].set_title("Rössler attractor — x component (chaotic, dt=0.02)")
axes[0].set_xlabel("virtual time"); axes[0].set_ylabel("x(t)")

axes[1].semilogy(freqs_r, psd_r, color="purple")
axes[1].set_title("Welch PSD — Rössler x")
axes[1].set_xlabel("frequency (Hz)"); axes[1].set_ylabel("PSD (log scale)")

fig.suptitle("Section 9 — Rössler Attractor", fontweight="bold")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "09_rossler_signal.png", dpi=120, bbox_inches="tight")
plt.show()

In [95]:
# ── 9.2  Offline SSD on Rössler ─────────────────────────────────────────────

eng_ross = SSD(fs=fs_ross, nmse_threshold=0.05, max_iter=10)
comps_ross = eng_ross.fit(sig_ross)
comps_ross_only, res_ross = comps_ross[:-1], comps_ross[-1]
recon_ross = np.sum(comps_ross_only, axis=0)

print(f"Rössler SSD: {len(comps_ross_only)} components  "
      f"NMSE={nmse(sig_ross-recon_ross, sig_ross):.5f}")

for ci, comp in enumerate(comps_ross_only):
    f_dom = dominant_frequency(comp, fs=fs_ross)
    print(f"  Component {ci}: dominant freq ≈ {f_dom:.3f} Hz")

plot_decomposition(
    sig_ross, comps_ross_only, residual=res_ross, fs=fs_ross,
    title="Section 9 — SSD Decomposition of Rössler Attractor",
    save_path=str(RESULTS_DIR / "09_rossler_ssd.png"),
)
print("Saved: 09_rossler_ssd.png")

Rössler SSD: 1 components  NMSE=0.00354
  Component 0: dominant freq ≈ 0.195 Hz
Saved: 09_rossler_ssd.png


In [96]:
# ── 9.3  Streaming pipeline on Rössler ──────────────────────────────────────

WLEN_R, STRIDE_R = 200, 100
wm_r  = WindowManager(window_len=WLEN_R, stride=STRIDE_R, fs=fs_ross)
eng_r = SSD(fs=fs_ross, nmse_threshold=0.05, max_iter=8)
mat_r = ComponentMatcher(distance="d_freq", freq_weight=1.0, fs=fs_ross,
                          lookback=5, max_cost=0.2, max_trajectories=6)
str_r = TrajectoryStore(max_components=6, max_len=N_ROSS)

for si, x in enumerate(sig_ross):
    w = wm_r.push(float(x))
    if w is None:
        continue
    comps_w = eng_r.fit(w)[:-1]
    mapping_w = mat_r.match_stateful(comps_w, wm_r.overlap)
    str_r.update(si - WLEN_R + 1, comps_w, mapping_w, wm_r.overlap)

trajs_r = str_r.get_all()
print(f"Rössler streaming: {len(trajs_r)} trajectories")

plot_trajectory_overlay(str_r, sig_ross, fs=fs_ross,
    save_path=str(RESULTS_DIR / "09_rossler_trajectories.png"))
print("Saved: 09_rossler_trajectories.png")

Rössler streaming: 4 trajectories
Saved: 09_rossler_trajectories.png


In [97]:
# ── 9.4  Dynamic component onset detection ──────────────────────────────────

ONSET_N   = 4000
ONSET_SAM = 2000
sig_onset = component_onset(
    N=ONSET_N, f_steady=30.0, f_onset=120.0,
    onset_sample=ONSET_SAM, fs=FS, seed=SEED,
)

wm_on  = WindowManager(window_len=400, stride=200, fs=FS)
eng_on = SSD(fs=FS, nmse_threshold=NMSE_THR, max_iter=6)
mat_on = ComponentMatcher(distance="d_freq", freq_weight=1.0, fs=FS,
                           lookback=10, max_cost=0.10, max_trajectories=4)
str_on = TrajectoryStore(max_components=4, max_len=ONSET_N)

active_log = []
for si, x in enumerate(sig_onset):
    w = wm_on.push(float(x))
    if w is None:
        continue
    comps_w = eng_on.fit(w)[:-1]
    mapping_w = mat_on.match_stateful(comps_w, wm_on.overlap)
    str_on.update(si - wm_on.window_len + 1, comps_w, mapping_w, wm_on.overlap)
    active_log.append((si / FS,
                        len(set(mapping_w.values()) - {ComponentMatcher.DROP_ID})))

tt_on, cc_on = zip(*active_log)
fig, ax = plt.subplots(figsize=(11, 3))
ax.step(tt_on, cc_on, where="post", color="purple", linewidth=1.5)
ax.axvline(ONSET_SAM / FS, color="red", linestyle="--",
            label=f"onset at t={ONSET_SAM/FS:.1f} s")
ax.set_title("Dynamic component onset detection — active trajectory count")
ax.set_xlabel("time (s)"); ax.set_ylabel("# active trajectories")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "09_onset_detection.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: 09_onset_detection.png")

Saved: 09_onset_detection.png


In [98]:
# ── 9.5  Experiment runner integration ──────────────────────────────────────

import yaml
with open("../experiments/configs/baseline.yaml") as fh:
    baseline_cfg = yaml.safe_load(fh)
print("baseline.yaml:"); print(yaml.dump(baseline_cfg, sort_keys=False))

# build_pipeline
cfg_copy = yaml.safe_load(yaml.dump(baseline_cfg))
wm_bp, eng_bp, mat_bp, str_bp = build_pipeline(cfg_copy,
                                                  signal_length=baseline_cfg["signal"]["N"])
print("build_pipeline:", type(wm_bp).__name__, type(eng_bp).__name__,
      type(mat_bp).__name__, type(str_bp).__name__)

# Full experiment run
exp_cfg = yaml.safe_load(yaml.dump(baseline_cfg))
exp_out = str(RESULTS_DIR / "experiment_run")
run_experiment(config_dict=exp_cfg, output_dir=exp_out)

exp_metrics = pd.read_csv(Path(exp_out) / "metrics.csv")
print(f"\nExperiment metrics.csv: {exp_metrics.shape[0]} windows")
print(exp_metrics[["window_index", "qrf", "matching_confidence"]].head())

baseline.yaml:
signal:
  type: chirp_plus_sinusoid
  N: 3000
  fs: 1000.0
  f_sin: 50.0
  f_start: 10.0
  f_end: 150.0
  snr_db: 20.0
streaming:
  window_len: 300
  stride: 150
  max_components: 100
engine:
  name: ssd
  nmse_threshold: 0.01
  max_iter: 20
matcher:
  distance: d_freq
  freq_weight: 1.0
  lookback: 10
  max_cost: 0.1
output:
  save_trajectories: true
  save_metrics: true

build_pipeline: WindowManager SSD ComponentMatcher TrajectoryStore


Streaming: 100%|██████████| 3000/3000 [00:00<00:00, 4352.58it/s]


Experiment metrics.csv: 19 windows
   window_index        qrf  matching_confidence
0             0  20.351939                  NaN
1             1  21.611801             0.986667
2             2  23.481225             1.000000
3             3  21.932896             0.993333
4             4  20.496243             0.995556


## Section 10 — Summary & Reproducibility

The table below lists every public symbol demonstrated in this notebook, the algorithm
behind it, and the corresponding paper reference. All cells above are independently
re-runnable after *Kernel → Restart & Run All*.

| Module | Symbol | Algorithm / Paper |
|--------|---------|------------------|
| `generators` | `two_sinusoids`, `chirp_plus_sinusoid`, `rossler`, `component_onset`, `n_sinusoids` | synthetic test signals |
| `engines` | `DecompositionEngine`, `get_engine`, `_REGISTRY` | factory pattern |
| `engines/ssd.py` | `SSD.fit`, `_build_trajectory_matrix`, `_choose_window_length`, `_reconstruct_component` | Bonizzi et al. 2014 |
| `engines/ssa.py` | `build_trajectory_matrix`, `svd_decompose`, `diagonal_averaging`, `auto_ssa`, `SSA` | Golyandina et al. SSA |
| `engines/rsvd.py` | `rsvd` | Halko, Martinsson & Tropp 2011 |
| `engines/ssd_incremental.py` | `IncrementalSSD` (cold/warm/rSVD) | warm-start SVD caching |
| `engines/ssd_optimized.py` | `OptimizedSSD` (`fwhm`, `moment`, `gaussian`) | Rainio et al. FWHM 2025 |
| `engines/svd_update.py` | `RankOneUpdater` (`.update`, `.slide_window`), `_build_hankel` | Brand 2003; Saeed et al. 2020 |
| `engines/ssd_rank1.py` | `RankOneIncrementalSSD` | USSA (Saeed 2020) |
| `metrics/similarity.py` | `d_corr`, `d_freq`, `w_correlation`, `subspace_angle` | Harmouche 2017 |
| `metrics/stability.py` | `qrf`, `nmse`, `energy_continuity`, `singular_value_drift`, `dominant_frequency`, `freq_drift_aggregate`, `matching_confidence` | Harmouche 2017 |
| `streaming/window_manager.py` | `WindowManager` (`.push`, `.overlap`, `.reset`) | sliding-window framework |
| `streaming/component_matcher.py` | `ComponentMatcher` (all 3 distances, `max_cost`, `max_trajectories`, `lookback`, stateful/stateless API) | Hungarian algorithm |
| `streaming/trajectory_store.py` | `TrajectoryStore` (`.update`, `.get`, `.get_all`, online averaging) | this work |
| `visualization/` | `plot_decomposition`, `plot_trajectory_overlay`, `plot_component_spectra`, `plot_matching_graph`, `plot_metrics_over_windows`, `plot_pipeline_dashboard`, `plot_window_reconstruction`, `plot_window_grid`, `plot_nmse_over_time`, `plot_metrics` | all 10 functions |
| `run_experiment.py` | `build_pipeline`, `run` | YAML-driven experiment runner |

In [99]:
# ── 10.1  Total notebook execution time ────────────────────────────────────

_elapsed_total = time.perf_counter() - _T0_NOTEBOOK
mins, secs = divmod(_elapsed_total, 60)
print(f"Total notebook execution time: {int(mins):02d}m {secs:.1f}s")
print()
print("Saved figures:")
for p in sorted(RESULTS_DIR.glob("*.png")):
    print(f"  {p.name}")

Total notebook execution time: 00m 23.5s

Saved figures:
  01_signals.png
  02_component_spectra.png
  02_reconstruction.png
  02_ssd_decomp.png
  02_svd_spectrum.png
  02_traj_matrix.png
  03_rsvd_k_comparison.png
  03_rsvd_speedup.png
  04_naive_sliding.png
  05_matching_graph.png
  05_traj_comparison.png
  06_incremental_accuracy.png
  06_sv_tracking.png
  06_timing_comparison.png
  07_metrics_over_windows.png
  07_nmse_over_time.png
  07_pipeline_dashboard.png
  07_streaming_pipeline.png
  07_traj_overlay.png
  07_win_first.png
  07_win_last.png
  07_win_mid.png
  07_window_grid.png
  08_benchmark.png
  08_snr_sweep.png
  09_onset_detection.png
  09_rossler_signal.png
  09_rossler_ssd.png
  09_rossler_trajectories.png
  metrics_plot.png
